In [ ]:
import deepfmkit.physics as physics
from deepfmkit.fitters import StandardNLSFitter, EKFFitter
import numpy as np
from tqdm import tqdm

from deepfmkit.plotting import default_rc, cmap_parula
import matplotlib.pyplot as plt

plt.rcParams.update(default_rc)

In [ ]:
f_samp = 200e3
fm = 1e3
phi_0 = np.random.uniform(-np.pi, np.pi)
psi_0 = np.random.uniform(-np.pi, np.pi)
m_0 = 5.678901
n_fit = int(1)
N = int(m_0 + 10)

laser = physics.LaserConfig()
laser.fm = fm

ifo = physics.IfoConfig()
main = physics.SimConfig(
    label="Hello World!", laser_config=laser, ifo_config=ifo, f_samp=f_samp
)
sg = physics.SignalGenerator()

laser.set_df_for_effect(ifo, m_0)

In [ ]:
dB_list = np.linspace(20, 120, 40)

colors = cmap_parula(np.linspace(0, 1, len(dB_list)))

fig, ax = plt.subplots(figsize=(4, 2), dpi=150)
for i, dB in enumerate(dB_list):
    raw = sg.generate(main, n_seconds=n_fit / fm, mode="snr", snr_db=dB, trial_num=3)[
        "main"
    ]
    ax = raw.plot(ax=ax, ls="--", lw=0.8, color=colors[i])
    ax.grid(False)

plt.show()

In [ ]:
nls = StandardNLSFitter(
    {"n": n_fit, "ndata": N, "init_m": m_0, "init_phi": phi_0, "init_psi": psi_0}
)
ekf = EKFFitter({"n": n_fit, "init_m": m_0, "init_phi": phi_0, "init_psi": psi_0})

print(nls.fit(raw, parallel=False))
print(ekf.fit(raw, verbose=False))

In [ ]:
# Calculate R (raw samples per fit buffer) based on the sampling frequency
B = int(n_fit * f_samp / laser.fm)

# Calculate the actual simulation time in seconds.
n_seconds_to_simulate = B / f_samp

In [ ]:
from joblib import Parallel, delayed

num_trials = 100


def process_trial(dB, trial_num):
    raw = sg.generate(
        main, n_seconds=n_fit / fm, mode="snr", snr_db=dB, trial_num=trial_num
    )["main"]
    df_nls = nls.fit(raw, parallel=False)
    df_ekf = ekf.fit(raw, verbose=False)
    iloc = -1
    return (
        df_nls["m"].iloc[iloc],
        df_ekf["m"].iloc[iloc],
        df_nls["phi"].iloc[iloc],
        df_ekf["phi"].iloc[iloc],
    )


m_nls_result = []
m_ekf_result = []
m_nls_var = []
m_ekf_var = []

phi_nls_result = []
phi_ekf_result = []
phi_nls_var = []
phi_ekf_var = []

for dB in tqdm(dB_list):
    results = Parallel(n_jobs=-1, backend="loky")(
        delayed(process_trial)(dB, j) for j in range(num_trials)
    )
    m_nls_result.append([r[0] for r in results])
    m_ekf_result.append([r[1] for r in results])
    phi_nls_result.append([r[2] for r in results])
    phi_ekf_result.append([r[3] for r in results])
    m_nls_var.append(np.var(m_nls_result[-1]))
    m_ekf_var.append(np.var(m_ekf_result[-1]))
    phi_nls_var.append(np.var(phi_nls_result[-1]))
    phi_ekf_var.append(np.var(phi_ekf_result[-1]))

In [ ]:
SNR_lin = 10 ** (dB_list / 20)
m_CRLB = 4 / (B * SNR_lin**2)

fig, ax = plt.subplots(figsize=(3.375, 1.8))

ax.semilogy(dB_list, m_CRLB, label=r"$m$ CRLB", c="gray", lw=4)
ax.semilogy(dB_list, np.array(m_nls_var), label="NLS", c="#00BFFF", lw=2)
ax.semilogy(dB_list, np.array(m_ekf_var), label="EKF", ls="--", c="#FFA07A", lw=2)

ax.legend(edgecolor="k", framealpha=1)
ax.set_xlabel("Voltage SNR (dB)")
ax.set_ylabel(r"$\sigma_m^2$")
fig.tight_layout()
plt.show()

In [ ]:
from scipy.special import jv

SNR_lin = 10 ** (dB_list / 20)
phi_CRLB = 2 / (B * SNR_lin**2 * (1 - (jv(0, 6.0)) ** 2))

fig, ax = plt.subplots(figsize=(3.375, 1.8))

ax.semilogy(dB_list, phi_CRLB, label=r"$\phi$ CRLB", c="gray", lw=4)
ax.semilogy(dB_list, np.array(phi_nls_var), label="NLS", c="#00BFFF", lw=2)
ax.semilogy(dB_list, np.array(phi_ekf_var), label="EKF", ls="--", c="#FFA07A", lw=2)

ax.legend(edgecolor="k", framealpha=1)
ax.set_xlabel("Voltage SNR (dB)")
ax.set_ylabel(r"$\sigma_{\phi}^2$")
fig.tight_layout()
plt.show()